# WILO — Qwen 2.5 3B Inference

Run on RunPod with a RTX 3090 or 4090 (24GB VRAM).

**Before running:** set your HuggingFace token in the cell below.

In [ ]:
# Install dependencies (torch is pre-installed in RunPod template)
!pip install transformers huggingface_hub python-dotenv accelerate pandas -q

import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none'}")

In [ ]:
# HuggingFace login — paste your token here
from huggingface_hub import login
login(token="hf_your_token_here")

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

MODEL_ID = "Qwen/Qwen2.5-3B-Instruct"

print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

print("Loading model...")
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, dtype=torch.float16, device_map="auto")
model.eval()
print(f"Model loaded on: {model.device}")

In [ ]:
# Load workout history
import pandas as pd
import json
import glob

df = pd.read_csv("/workspace/workout_data.csv")
df = df[["exercise_title", "start_time", "set_index", "weight_lbs", "reps", "exercise_notes"]].copy()
df = df[df["weight_lbs"].notna() | df["reps"].notna()]  # drop empty sets

summary_lines = []
for exercise, group in df.groupby("exercise_title"):
    recent = group.sort_values("start_time", ascending=False).head(5)
    lines = recent[["start_time", "set_index", "weight_lbs", "reps"]].to_string(index=False)
    # include notes if any exist for this exercise
    notes = group["exercise_notes"].dropna().unique()
    notes = [n for n in notes if n.strip()]
    note_str = f"\nNotes: {'; '.join(notes)}" if notes else ""
    summary_lines.append(f"### {exercise}{note_str}\n{lines}")
csv_summary = "\n\n".join(summary_lines)

json_parts = []
for path in sorted(glob.glob("/workspace/sess_*.json")):
    with open(path) as f:
        json_parts.append(f"--- {path} ---\n{f.read()}")
json_summary = "\n\n".join(json_parts) if json_parts else "No WILO session files found."

SYSTEM_PROMPT = f"""You are a personal strength coach. You have full access to the user's workout history.

## Hevy workout history (last 5 sets per exercise, most recent first):
{csv_summary}

## Recent WILO session programs:
{json_summary}

Your job: analyze the history, identify patterns, and when asked, design new workout programs in the same style as the WILO sessions. Be specific with sets, reps, and weights based on what the user has actually been doing."""

print("Workout data loaded.")
print(f"Exercises tracked: {df['exercise_title'].nunique()}")
print(f"Approx system prompt tokens: {len(SYSTEM_PROMPT.split())}")

In [ ]:
history = [{"role": "system", "content": SYSTEM_PROMPT}]

while True:
    prompt = input("You: ").strip()
    if not prompt or prompt.lower() in ("exit", "quit"):
        break
    history.append({"role": "user", "content": prompt})
    text = tokenizer.apply_chat_template(history, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors="pt").to(model.device)
    with torch.no_grad():
        output = model.generate(**inputs, max_new_tokens=500)
    reply = tokenizer.decode(output[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    history.append({"role": "assistant", "content": reply})
    print(f"Model: {reply}\n")